# AutoStock — Daily Screening Notebook

This notebook:
1. Loads the ~500 relevant tickers from Finnhub (cached)
2. Lets you add/remove tickers manually
3. Incrementally updates price data (only fetches new bars)
4. Runs all trading strategies and displays triggered signals

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

from tickers        import get_tickers
from get_stock_data import update_all_stocks, get_stock_data, last_updated, _load
from strategy       import check_all_strategies, STRATEGIES
from information    import run_news_discovery

PROJECT_DIR = Path(os.path.abspath('__file__')).parent
ALARMS_DIR  = PROJECT_DIR / 'alarms'
ALARMS_DIR.mkdir(exist_ok=True)

# Alarm file for today's run
run_ts   = datetime.now(tz=timezone.utc)
run_label = run_ts.strftime('%Y-%m-%d_%H-%M')
alarm_path = ALARMS_DIR / f'{run_label}.txt'

print(f'Run timestamp : {run_ts.strftime("%Y-%m-%d %H:%M UTC")}')
print(f'Alarm file    : {alarm_path}')

## 1 — Ticker Universe

In [ ]:
# ── Step 1: Discover new tickers from recent news ────────────────────────────
print('Running news discovery (Finnhub + LLM) ...')
newly_added = run_news_discovery(alarm_path=alarm_path)
if newly_added:
    print(f'  News discovered {len(newly_added)} new ticker(s): {newly_added}')
else:
    print('  No new tickers discovered from news.')

# Reload tickers (includes any newly discovered ones)
from importlib import reload
import tickers as _tickers_mod
reload(_tickers_mod)
from tickers import get_tickers

base_tickers = get_tickers(include_etfs=False)
print(f'Base list: {len(base_tickers)} tickers (including {len(_tickers_mod.NEWS_DISCOVERED)} news-discovered)')

## 2 — Manual Watchlist (edit freely)

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# ADD or REMOVE tickers here.  Changes only affect this run — they are NOT
# saved back to the Finnhub cache.
# ─────────────────────────────────────────────────────────────────────────────

manually_added = [
    # 'PLTR', 'ARM', 'IONQ',   # ← example: add tickers you follow
]

manually_removed = [
    # 'BRK.B',                  # ← example: drop tickers you don't want
]

# ─── Merge ───────────────────────────────────────────────────────────────────
all_tickers = list(dict.fromkeys(
    [t for t in base_tickers + manually_added
     if t not in manually_removed]
))

print(f'Final universe: {len(all_tickers)} tickers')
print('First 20:', all_tickers[:20])

Final universe: 354 tickers
First 20: ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'GOOG', 'AMZN', 'META', 'TSLA', 'AVGO', 'ORCL', 'ADBE', 'CRM', 'IBM', 'SAP', 'INTU', 'AMD', 'INTC', 'QCOM', 'TXN', 'MU']


## 3 — Update Price Data (incremental)

In [ ]:
today = datetime.now(tz=timezone.utc).date()
needs_update, status_rows = [], []
for t in all_tickers:
    lu = last_updated(t)
    if lu is None or lu.date() < today:
        needs_update.append(t)
    status_rows.append({'ticker': t, 'last_updated': lu.strftime('%Y-%m-%d %H:%M UTC') if lu else 'never'})

df_status = pd.DataFrame(status_rows)
print(f'{len(needs_update)} of {len(all_tickers)} tickers need a data update.')
print(f'\nData freshness sample (first 10):')
print(df_status.head(10).to_string(index=False))

In [ ]:
# ── Incremental download ──────────────────────────────────────────────────────
# Only fetches new bars since the last stored date.
# Set force_full=True to re-download full history for all tickers.
stock_data = update_all_stocks(needs_update, force_full=False)

# Also load already-up-to-date tickers from disk
for t in all_tickers:
    if t not in stock_data:
        df = get_stock_data(t)
        if df is not None and not df.empty:
            stock_data[t] = df

print(f'Data available for {len(stock_data)} tickers.')

## 4 — Run Strategies

In [ ]:
results_all = []

for sym, df in stock_data.items():
    any_triggered, strat_results = check_all_strategies(df, sym)
    if any_triggered:
        for r in strat_results:
            if r['triggered']:
                results_all.append({
                    'Symbol'  : sym,
                    'Strategy': r['strategy'],
                    'Signal'  : r['description'],
                })

print(f'Total signals across all stocks: {len(results_all)}')

In [ ]:
# ── Save signals + data timestamps to alarm file ─────────────────────────────
ALARMS_DIR.mkdir(exist_ok=True)

# Find latest bar date per ticker
latest_dates = {sym: df.index.max().strftime('%Y-%m-%d') for sym, df in stock_data.items()}
overall_latest = max(latest_dates.values()) if latest_dates else 'N/A'

with open(alarm_path, 'w', encoding='utf-8') as f:
    f.write(f'AutoStock Daily Report\n')
    f.write(f'Generated : {run_ts.strftime("%Y-%m-%d %H:%M UTC")}\n')
    f.write(f'Latest data: {overall_latest} (most recent bar across all tickers)\n')
    f.write(f'Universe   : {len(stock_data)} tickers\n')
    f.write('=' * 60 + '\n')
    f.write('SIGNALS\n')
    f.write('=' * 60 + '\n')
    if not results_all:
        f.write('  No signals triggered today.\n')
    else:
        for strat_name in [s[0] for s in STRATEGIES]:
            subset = [r for r in results_all if r['Strategy'] == strat_name]
            if not subset:
                continue
            f.write(f'\n[{strat_name}]\n')
            for r in subset:
                data_date = latest_dates.get(r['Symbol'], '?')
                f.write(f"  {r['Symbol']:8s} (data thru {data_date})\n")
                f.write(f"  {r['Signal']}\n\n")

print(f'Alarm saved → {alarm_path}')
print(f'Latest stock data: {overall_latest}')
print(f'Total signals    : {len(results_all)}')

# Print the file contents
print('\n' + '─'*60)
print(alarm_path.read_text())

## 5 — Display Results

In [ ]:
if not results_all:
    print('No trading signals triggered today.')
else:
    df_signals = pd.DataFrame(results_all)
    
    # ── Summary table ─────────────────────────────────────────────────────────
    summary = (
        df_signals.groupby(['Symbol', 'Strategy'])
        .size()
        .reset_index(name='Count')
        .sort_values('Symbol')
    )
    print(f'Stocks with signals: {df_signals["Symbol"].nunique()}')
    print()
    
    # Style the table
    styled = (
        df_signals
        .style
        .set_table_styles([{
            'selector': 'th',
            'props': [('background-color', '#2c3e50'),
                      ('color', 'white'),
                      ('font-weight', 'bold'),
                      ('padding', '8px')]
        }, {
        'selector': 'td',
            'props': [('padding', '6px 12px'),
                      ('border-bottom', '1px solid #ddd')]
        }])
        .apply(lambda x: ['background-color: #eafaf1' if x.name % 2 == 0
                           else 'background-color: #fdfefe'
                           for _ in x], axis=1)
        .set_caption(f'AutoStock Signals — {today}')
    )
    display(styled)

In [ ]:
# ── Per-strategy breakdown ─────────────────────────────────────────────────────
if results_all:
    df_signals = pd.DataFrame(results_all)
    strategy_names = [s[0] for s in STRATEGIES]
    for strat in strategy_names:
        subset = df_signals[df_signals['Strategy'] == strat]
        if subset.empty:
            continue
        print(f'\n━━ {strat} ({len(subset)} signal(s)) ━━')
        for _, row in subset.iterrows():
            print(f"  {row['Symbol']:8s}  {row['Signal']}")

## 6 — Optional: Drill into a single stock

In [ ]:
INSPECT = 'NVDA'   # ← change to any ticker

if INSPECT in stock_data:
    df_inspect = stock_data[INSPECT]
    cols_show = ['Close', 'MA20', 'MA50', 'MA200',
                 'DIF', 'DEA', 'MACD_hist', 'RSI14',
                 'BB_upper', 'BB_lower', 'ATR14', 'Volume']
    print(f'{INSPECT} — last 10 bars')
    display(df_inspect[cols_show].tail(10).round(4))
    
    print()
    _, strat_results = check_all_strategies(df_inspect, INSPECT)
    for r in strat_results:
        status = '✓ TRIGGERED' if r['triggered'] else '✗'
        print(f"  {status:15s} [{r['strategy']}]  {r['description']}")
else:
    print(f'{INSPECT} not in loaded dataset.')